In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# Import Data

In [ ]:
nadp_weekly = pd.read_csv('/Users/beneck/Library/CloudStorage/OneDrive-NortheasternUniversity/Boise Project/Data/Reynolds Nitrate Monitoring/NADP Data/NTN-id11-w-s-mg.csv', parse_dates=True, na_values=-9, index_col = 'dateOn')
nadp_weekly

In [ ]:
rmsp3_combined = pd.read_csv('/Users/beneck/Library/CloudStorage/OneDrive-NortheasternUniversity/Boise Project/Data/Reynolds Nitrate Monitoring/USDA Data/2025 Preliminary Data/Processed Data/rmsp3combined.csv', index_col='datetime', parse_dates=True)
combined_176 = pd.read_csv('/Users/beneck/Library/CloudStorage/OneDrive-NortheasternUniversity/Boise Project/Data/Reynolds Nitrate Monitoring/USDA Data/2025 Preliminary Data/Processed Data/176combined.csv', index_col='datetime', parse_dates=True)

combined_176

In [ ]:
fig, ax = plt.subplots(nrows = 2, figsize=(11,8.5), sharex=True)
nadp_weekly.loc['11/1/2024':].reset_index().plot(x='dateOff', y='NO3', kind='bar', ylim=(0,1.2),ax=ax[0])
nadp_weekly.loc['11/1/2024':].reset_index().plot(x='dateOff', y='ppt', kind='bar',ax=ax[1])

fig.tight_layout()

In [ ]:
combined_176

# Extract Valid Weeks

In [ ]:
# filter out invalid invalcodes
nadp_weekly_valid = nadp_weekly.loc[~(nadp_weekly['invalcode'].str.contains('b|u|f|c|v|e|l|i|n|p|x', regex=True))]

# keep only valid valcodes
nadp_weekly_valid = nadp_weekly_valid.loc[nadp_weekly_valid['valcode'].str.contains('w')]

nadp_weekly_valid['valcode'].unique()

In [ ]:
nadp_weekly_valid['invalcode'].unique()

In [ ]:
fig, ax = plt.subplots(nrows = 2, figsize=(11,8.5), sharex=True)
nadp_weekly_valid.loc['11/1/2024':].reset_index().plot(x='dateOff', y='NO3', kind='bar', ylim=(0,1.2),ax=ax[0])
nadp_weekly_valid.loc['11/1/2024':].reset_index().plot(x='dateOff', y='ppt', kind='bar',ax=ax[1])

fig.tight_layout()

In [ ]:
nadp_weekly.columns

In [ ]:
jan = nadp_weekly_valid['2025-01-01':'2025-01-31']

In [ ]:
numeric_cols = ['ph', 'Conduc', 'Ca', 'Mg', 'K', 'Na', 'NH4', 'NO3', 'Cl', 'SO4', 'Br', 'svol', 'ppt', 'subppt']

In [ ]:
def calc_weighted_mean(df, weight):
    if df[weight].sum() <= 0:
        return pd.Series([np.nan] * len(df.columns), index=df.columns)

    else:
        result = np.average(df, axis=0, weights=df[weight])
        return pd.Series(result, index = df.columns)


nadp_monthly= nadp_weekly_valid[numeric_cols].groupby(pd.Grouper(freq='ME')).apply(lambda x: calc_weighted_mean(x, 'subppt'))
#monthly_grouped = pd.DataFrame(data = monthly_grouped_array[1],index=monthly_grouped_array[0], columns = numeric_cols)
nadp_monthly

# Merge with Precip and Calculate Concentration

In [ ]:
nadp_hourly = nadp_monthly.resample('1h').bfill()
nadp_hourly

In [ ]:
rmsp3_combined = pd.merge(rmsp3_combined, nadp_hourly, right_index=True, left_index=True, how='outer')
combined_176 = pd.merge(combined_176, nadp_hourly, right_index=True, left_index=True, how='outer')


In [ ]:
def calculate_deposition(df, precip_type, precip_col, analytes):
    for analyte in analytes:
        dep_col = precip_type + ' ' + analyte + ' Deposition'
        df[dep_col]  = df[precip_col] * df[analyte]/100
        cum_dep_col = 'Cumulative ' + precip_type + ' ' + analyte + ' Deposition'
        df[cum_dep_col] = df.groupby('water_year').transform(lambda x: x.cumsum())[dep_col]
        df['Snowpack ' + analyte+ ' Concentration'] = df[cum_dep_col]/df['cum_swe']*100
    return df

In [ ]:
rmsp3_combined = calculate_deposition(rmsp3_combined, 'Snow', 'ppta_snow', ['NO3', 'NH4'])
combined_176 = calculate_deposition(combined_176, 'Snow', 'ppta_snow', ['NO3', 'NH4'])
rmsp3_combined['Cumulative Snow N Deposition']  = rmsp3_combined['Cumulative Snow NO3 Deposition'] + rmsp3_combined['Cumulative Snow NH4 Deposition']
combined_176['Cumulative Snow N Deposition']  = combined_176['Cumulative Snow NO3 Deposition'] + combined_176['Cumulative Snow NH4 Deposition']

In [ ]:
rme_snow = pd.read_csv('/Users/beneck/Library/CloudStorage/OneDrive-NortheasternUniversity/Boise Project/Data/Reynolds Nitrate Monitoring/Nutrient Analyzer Data/Processed Data/rme_snow_aa500.csv',index_col = 'Sample Datetime',parse_dates=True)
rme_snow

In [ ]:
rme_avg = rme_snow['Nitrate mean'].describe()
rme_avg = pd.DataFrame([rme_avg], index = [pd.to_datetime('2025-04-02')])
rme_avg.index.name = 'Sample Datetime'
rme_avg['err'] = 2* rme_avg['std']
rme_avg


In [ ]:
rme_avg_nh4 = rme_snow['Ammonium mean'].describe()
rme_avg_nh4 = pd.DataFrame([rme_avg_nh4], index = [pd.to_datetime('2025-04-02')])
rme_avg_nh4.index.name = 'Sample Datetime'
rme_avg_nh4['err'] = 2* rme_avg_nh4['std']
rme_avg_nh4

In [ ]:
fig, ax = plt.subplots(nrows=4, figsize = (11,8.5))

start_date = '10/01/2024'
end_date = '05/12/2025'

second_y = ax[0].twinx()
third_y = ax[0].twinx()

combined_176.loc[start_date:end_date].plot(y='sno', ax=ax[0], x_compat=True, label='Snow Depth', ylabel = 'Snow Depth (cm)', title= '2025 WY Snowpack Development at RME Ridgetop', color='lightsteelblue')
combined_176.loc[start_date:end_date].plot(y='cum_swe', ax=third_y, x_compat=True, label='Cumulative Snowfall WE (mm)', ylabel = 'Cumulative Snowfall WE (mm)', legend=None, color='cornflowerblue')

second_y.bar(combined_176.loc[start_date:end_date].index,combined_176.loc[start_date:end_date]['ppta_rain'], width = pd.Timedelta(1,'hour'),color='royalblue', label='Rain')
second_y.bar(combined_176.loc[start_date:end_date].index,combined_176.loc[start_date:end_date]['ppta_snow'], width = pd.Timedelta(1,'hour'),color='tab:orange', label='Snow')
second_y.set_ylim(0,10)
ax[0].set_ylim(-5, 250)
second_y.set_ylabel('Hourly Precipitation (mm)')
second_y.invert_yaxis()
third_y.spines['right'].set_position(('axes', 1.1)) # Adjust the '1.15' value as needed

# Combine legends from both axes
handles1, labels1 = ax[0].get_legend_handles_labels()  # Primary y-axis
handles2, labels2 = second_y.get_legend_handles_labels()  # Secondary y-axis
handles3, labels3 = third_y.get_legend_handles_labels()
handles = handles1 + handles2 + handles3
labels = labels1 + labels2+ labels3

# Add the combined legend
ax[0].legend(handles, labels, loc='upper left')

#rmsp3_combined.loc[start_date:end_date].plot(y='sno', ax=ax[1], x_compat=True, label='Snow Depth', ylabel = 'Snow Depth (cm)', title='RMSP3 Snowpack Development')
#second_y2.bar(rmsp3_combined.loc[start_date:end_date].index,rmsp3_combined.loc[start_date:end_date]['ppta_rain'], width = pd.Timedelta(1,'hour'),color='royalblue', label='Rain')
#second_y2.bar(rmsp3_combined.loc[start_date:end_date].index,rmsp3_combined.loc[start_date:end_date]['ppta_snow'], width = pd.Timedelta(1,'hour'),color='tab:orange', label='Snow')
#second_y2.set_ylim(0,10)
#ax[1].set_ylim(-5, 250)
#second_y2.set_ylabel('Hourly Precipitation (mm)')
#second_y2.invert_yaxis()

combined_176.loc[start_date:end_date].plot(y='Cumulative Snow NO3 Deposition', title='2025 WY Snow NO3 Deposition at RME Ridgetop', ax=ax[3], ylabel='Snow N Deposition (kg-N/ha)', x_compat=True)
combined_176.loc[start_date:end_date].plot(y='Cumulative Snow NH4 Deposition', title='2025 WY Snow NO3 Deposition at RME Ridgetop', ax=ax[3], ylabel='Snow N Deposition (kg-N/ha)', x_compat=True)
combined_176.loc[start_date:end_date].plot(y='Cumulative Snow N Deposition', title='2025 WY Snow NO3 Deposition at RME Ridgetop', ax=ax[3], ylabel='Snow N Deposition (kg-N/ha)', x_compat=True)

combined_176.loc[start_date:end_date].plot(y='NO3', ax=ax[1], ylabel='Monthly Average Precipitation Nitrate Concentration',  label = 'Precipitation NO3', x_compat=True)
combined_176.loc[start_date:end_date].plot(y='NH4', ax=ax[1], ylabel='Monthly Average Precipitation Nitrate Concentration',  label = 'Precipitation NH4', x_compat=True)

combined_176.loc[start_date:end_date].plot(y='Snowpack NO3 Concentration', title='2025 WY Snowpack Nitrate at RME Ridgetop', label = 'Average Snowpack NO3', ax=ax[2], ylabel=' NO3 Concentration (mg/L)',  x_compat=True)
combined_176.loc[start_date:end_date].plot(y='Snowpack NH4 Concentration', title='2025 WY Snowpack Nitrate at RME Ridgetop', label = 'Average Snowpack NH4', ax=ax[2], ylabel=' NO3 Concentration (mg/L)',  x_compat=True)

rme_avg.reset_index().plot(y='mean', x='Sample Datetime', kind='scatter', yerr='err',ax=ax[2], label = 'Measured Snowpack NO3', color = 'tab:green')
rme_avg_nh4.reset_index().plot(y='mean', x='Sample Datetime', kind='scatter', yerr='err',ax=ax[2], label = 'Measured Snowpack NH4', color='blue')


fig.tight_layout()